In [ ]:
# Add project root to path so `src` imports work from the notebook
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
# Generates realistic medical sample data without needing Telegram credentials.
# Populates: data/raw/telegram_messages/YYYY-MM-DD/{channel}.json
#            data/raw/images/{channel}/{message_id}.jpg
#            data/raw/csv/YYYY-MM-DD/telegram_data.csv
from src.scraper import run_demo

BASE_PATH = str(PROJECT_ROOT / "data")
LIMIT = 15  # messages per channel

channel_counts = run_demo(base_path=BASE_PATH, limit=LIMIT)
print("\nMessages scraped per channel:", channel_counts)

In [ ]:
# Only run this cell if you have Tg_API_ID and Tg_API_HASH in your .env file.
# This authenticates with Telegram and scrapes the actual channels.

# import asyncio
# from dotenv import load_dotenv
# from telethon import TelegramClient
# from src.scraper import scrape_all_channels, TARGET_CHANNELS

# load_dotenv()
# api_id   = int(os.getenv("Tg_API_ID"))
# api_hash = os.getenv("Tg_API_HASH")

# client = TelegramClient("telegram_scraper_session", api_id, api_hash)

# async def run():
#     async with client:
#         return await scrape_all_channels(client, TARGET_CHANNELS, BASE_PATH, limit=200)

# stats = asyncio.run(run())
# print(stats)

In [ ]:
# The manifest is an audit file written after each scrape run.
# It records which channels were scraped, how many messages each had, and when.
from src.datalake import list_available_dates, read_manifest

dates = list_available_dates(BASE_PATH)
print("Available date partitions:", dates)

latest = dates[-1]
manifest = read_manifest(BASE_PATH, latest)
print(f"\nManifest for {latest}:")
print(json.dumps(manifest, indent=2))

In [ ]:
# Reads each channel's JSON file from the data lake and shows a sample message.
from src.datalake import read_channel_messages_json
from datetime import datetime

TODAY = datetime.today().strftime("%Y-%m-%d")

for channel in ["lobelia4cosmetics", "tikvahpharma", "CheMed123", "DoctorsET"]:
    msgs = read_channel_messages_json(BASE_PATH, TODAY, channel)
    print(f"\n@{channel} — {len(msgs)} messages")
    if msgs:
        m = msgs[0]
        print(f"  Text:      {m['message_text'][:80]}...")
        print(f"  Date:      {m['message_date'][:19]}")
        print(f"  has_media: {m['has_media']} | views: {m['views']} | forwards: {m['forwards']}")

In [ ]:
# Loads the CSV backup into pandas for a quick overview of the full dataset.
import pandas as pd

csv_path = os.path.join(BASE_PATH, "raw", "csv", TODAY, "telegram_data.csv")
df = pd.read_csv(csv_path)

print(f"Total messages: {len(df)}\n")
print("Messages per channel:")
print(df["channel_name"].value_counts())
print("\nMedia breakdown:")
print(df["has_media"].value_counts())
print("\nViews summary:")
print(df["views"].describe().round(1))

In [ ]:
# Counts how many images were downloaded per channel.
# Images are stored at: data/raw/images/{channel_name}/{message_id}.jpg
image_root = os.path.join(BASE_PATH, "raw", "images")

print("Downloaded images per channel:")
for ch in sorted(os.listdir(image_root)):
    ch_path = os.path.join(image_root, ch)
    if os.path.isdir(ch_path):
        imgs = [f for f in os.listdir(ch_path) if f.endswith(".jpg")]
        print(f"  @{ch}: {len(imgs)} images")